In [56]:
# metaparameters
WINDOW_SIZE = 4
EMBEDDING_DIMENSION = 32
HIDDEN_LAYER_SIZE = 48
TRAINING_EPOCHS = 20

## Setup
* imports
* load shakespeare dataset: https://huggingface.co/datasets/sarnab/Shakespeare_Corpus
* generate training examples 

In [38]:
# imports & RNG
import random
import torch
import torch.nn as nn
import torch.optim as optim
from transformers import AutoTokenizer
from datasets import load_dataset

random.seed(42)
torch.manual_seed(42)


In [39]:
# load data
data = load_dataset("sarnab/Shakespeare_Corpus")
examples = data['train']['text']

In [40]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
pad_id = tokenizer.pad_token_id

def make_lm_examples(sentences, tokenizer, window_size=4):
    X = []
    y = []

    for sentence in sentences:
        ids = tokenizer(sentence, add_special_tokens=False)["input_ids"]
        for i in range(1, len(ids)):
            # Grab the recent context before the target token
            prefix = ids[max(0, i - window_size):i]
            # Add [PAD] tokens on the left if the context is shorter than the window
            prefix = [pad_id] * (window_size - len(prefix)) + prefix
            X.append(prefix)
            y.append(ids[i])

    return torch.tensor(X, dtype=torch.long), torch.tensor(y, dtype=torch.long)


In [41]:
X_train, y_train = make_lm_examples(examples, tokenizer, window_size=WINDOW_SIZE)
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("example input ids:", X_train[0])
print("example target id:", y_train[0].item())

Token indices sequence length is longer than the specified maximum sequence length for this model (594 > 512). Running this sequence through the model will result in indexing errors


X_train shape: torch.Size([281497, 4])
y_train shape: torch.Size([281497])
example input ids: tensor([   0,    0,    0, 2034])
example target id: 6926


## Define the RNN
we'll use the metaparameters we defined at the top of the file later; this is the same as what we did in class 

In [42]:
class TinyRNNLM(nn.Module):
    def __init__(self, vocab_size, embedding_dim=32, hidden_dim=64):
        super().__init__()

        # Look up a learned vector for each token id
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_id)

        # Read the sequence left-to-right, updating a hidden state at each step
        # batch_first=True shapes the data like this [batch_size, sequence_length, embedding_dim]
        self.rnn = nn.RNN(embedding_dim, hidden_dim, batch_first=True)

        # Turn the final hidden state into scores for every token in the vocabulary
        self.output = nn.Linear(hidden_dim, vocab_size)

    def forward(self, input_ids):
        # input_ids shape: [batch_size, sequence_length]
        emb = self.embedding(input_ids)

        # emb shape: [batch_size, sequence_length, embedding_dim]
        # hidden shape: [1, batch_size, hidden_dim]
        rnn_out, hidden = self.rnn(emb)

        # Remove the extra first dimension since we only have one RNN layer
        last_hidden = hidden.squeeze(0)

        # logits shape: [batch_size, vocab_size]
        logits = self.output(last_hidden)
        return logits



## Training Loop
* sends model & training examples to accelerator device
* batches input data

In [57]:
from torch.utils.data import TensorDataset, DataLoader

vocab_size = tokenizer.vocab_size
model = TinyRNNLM(vocab_size, embedding_dim=EMBEDDING_DIMENSION, hidden_dim=HIDDEN_LAYER_SIZE)

dataset = TensorDataset(X_train, y_train)
loader = DataLoader(dataset, batch_size=256, shuffle=True)

loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

# set the model to be in training mode
model.train()

device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using device: {device}")
model.to(device)

for epoch in range(TRAINING_EPOCHS):
    # We'll accumulate statistics across all batches so we can report one loss/accuracy per epoch
    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    # New idea: instead of pushing all training examples through the model at once,
    # the DataLoader gives us one small batch at a time
    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        # This part is the same as before, just on one batch instead of all of X_train / y_train
        logits = model(X_batch)
        loss = loss_fn(logits, y_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Keep track of how this batch did so we can average over the whole epoch
        batch_size = X_batch.size(0) # X_batch has shape [batch_size, sequence_length], so this gives us the batch_size
        total_loss += loss.item() * batch_size
        predictions = logits.argmax(dim=1)
        total_correct += (predictions == y_batch).sum().item()
        total_examples += batch_size

    if epoch % 1 == 0: 
        # Turn the accumulated totals into epoch-level averages
        avg_loss = total_loss / total_examples
        acc = total_correct / total_examples
        print(f"epoch {epoch+1}, loss={avg_loss:.4f}, acc={acc:.4f}")



Using device: cuda
epoch 1, loss=5.7561, acc=0.1454
epoch 2, loss=4.9609, acc=0.1966
epoch 3, loss=4.7327, acc=0.2060
epoch 4, loss=4.6004, acc=0.2133
epoch 5, loss=4.5134, acc=0.2179
epoch 6, loss=4.4468, acc=0.2228
epoch 7, loss=4.3962, acc=0.2266
epoch 8, loss=4.3543, acc=0.2300
epoch 9, loss=4.3261, acc=0.2322
epoch 10, loss=4.2964, acc=0.2340
epoch 11, loss=4.2737, acc=0.2358
epoch 12, loss=4.2594, acc=0.2376
epoch 13, loss=4.2449, acc=0.2383
epoch 14, loss=4.2332, acc=0.2390
epoch 15, loss=4.2239, acc=0.2411
epoch 16, loss=4.2163, acc=0.2410
epoch 17, loss=4.2133, acc=0.2419
epoch 18, loss=4.2062, acc=0.2431
epoch 19, loss=4.2009, acc=0.2435
epoch 20, loss=4.1998, acc=0.2441


## Text Generation


In [ ]:
unk_id = tokenizer.unk_token_id

def generate_text(start_text, model, tokenizer, num_tokens=8, window_size=4):

    # we're not training, so set the model to eval mode
    model.eval()
    generated_ids = tokenizer(start_text, add_special_tokens=False)["input_ids"]

    for _ in range(num_tokens):
        # Grab the recent context before the target token
        prefix = generated_ids[-window_size:]
        # Add [PAD] tokens on the left if the context is shorter than the window
        prefix = [pad_id] * (window_size - len(prefix)) + prefix
        x = torch.tensor([prefix], dtype=torch.long)
        x = x.to(device)

        with torch.no_grad():
            logits = model(x)
            next_id = logits.argmax(dim=1).item()

        generated_ids.append(next_id)

    tokens = tokenizer.convert_ids_to_tokens([i for i in generated_ids if i != pad_id and i != unk_id])
    return tokens

print(generate_text("All:", model, tokenizer))
print(generate_text("First", model, tokenizer))
print(generate_text("First Citizen:", model, tokenizer))
print(generate_text("Romeo", model, tokenizer, 32))
# these are the best values I could get, and they still aren't coherent at all
# this time it's still looping, but previous attempts (even with more training time/epochs) were much worse


['all', ':', 'i', 'have', 'forgot', 'a', 'buzz', '##ard', '-', 'den']
['first', 'keeper', ':', 'what', ',', 'i', 'have', 'forgot', 'a']
['first', 'citizen', ':', 'i', "'", 'll', 'not', 'be', 'avoided', 'of', 'the']
['romeo', ':', 'and', 'yet', 'i', 'have', 'forgot', ',', 'and', 'yet', 'i', 'have', 'forgot', ',', 'and', 'yet', 'i', 'have', 'forgot', ',', 'and', 'yet', 'i', 'have', 'forgot', ',', 'and', 'yet', 'i', 'have', 'forgot', ',', 'and']
